# DARF v6 — 100-Run Large-Scale Evaluation

**Goal:** Evaluate our best DARF v6 hair-refine method across:
- **4 reference image sets** (different photos of Hermione / Daenerys)
- **5 pose images** (face-aware dense pose variants)
- **5 prompt styles** (formal, royal/fantasy, modern casual, studio portrait, cinematic outdoor)
- **Total: 100 unique generations** with deterministic seeds

**Best config from v6:** `hair_refine_strength=0.55`, `lora_alpha=1.6`, `up_pad=1.6`

**Resume-safe:** skips runs where the final image + CSV row already exist. Eval-only if image exists but not yet recorded.

In [ ]:
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
# BASE_MODEL = "RunDiffusion/Juggernaut-XL-v9"

In [ ]:
import os
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/melik3kara/LoRA-scale-adaptive-feedback.git {REPO_DIR}
else:
    print("Pulling latest...")
    !cd {REPO_DIR} && git pull

!pip install -q gdown
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)
import glob
if not glob.glob(f"{REPO_DIR}/data/loras/*.safetensors"):
    !gdown --folder "https://drive.google.com/drive/folders/1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5" -O {REPO_DIR}/data/loras/
    import shutil
    for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
        target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
        if f != target:
            shutil.move(f, target)

os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)

!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu
!pip install -q "pydantic<2.10" "typing_extensions>=4.13.0" "mediapipe==0.10.14" "protobuf<5" scipy

os.chdir(REPO_DIR)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Base model: {BASE_MODEL}")

## Section 1 — Pipeline & Scorers

In [ ]:
from pipeline import load_identities, build_pipeline, generate_two_stage, face_local_refine, hair_local_refine
from scorer import FaceScorer
from pose_scorer import PoseScorer
from evaluator import Evaluator
from PIL import Image
from IPython.display import display
import pipeline as P
import numpy as np
import cv2

identities = load_identities()

# Hair descriptions enriched — same as v6 best run
identities["hermione"]["visual_description"] = (
    "front-facing young woman, long bushy brown curly hair, brunette, "
    "hazel eyes, fair skin, looking directly at the camera, black Hogwarts robe"
)
identities["daenerys"]["visual_description"] = (
    "front-facing woman, long straight platinum silver-white hair, icy blonde, "
    "violet eyes, pale skin, looking directly at the camera, regal blue dress"
)

identities["hermione"]["attention_phrases"] = [
    "long bushy brown curly hair", "brunette hair",
    "Hermione_Granger", "black Hogwarts robe",
]
identities["daenerys"]["attention_phrases"] = [
    "long straight platinum silver-white hair", "icy blonde silver hair",
    "dae woman", "regal blue dress",
]

identities["hermione"]["negative_features"] = (
    "blonde, silver hair, platinum hair, white hair, icy hair, crown, "
    "side profile, turned head"
)
identities["daenerys"]["negative_features"] = (
    "brown hair, brunette, dark hair, bushy curly hair, witch robe, school uniform, "
    "side profile, turned head"
)

P._BASE_NEGATIVE = (
    "blurry, low quality, deformed face, cartoon, illustration, anime, 3d render, "
    "three women, three people, third person, extra person, woman in background, "
    "side profile, profile view, turned head, looking sideways"
)

pipe = build_pipeline(identities, base_model=BASE_MODEL)

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
evaluator   = Evaluator(device="cuda")

names = list(identities.keys())
identity_regions = {names[0]: (0, 0, 512, 1024), names[1]: (512, 0, 1024, 1024)}

def detected_face_count(img):
    img_bgr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
    return len(evaluator.face_app.get(img_bgr))

print("Pipeline and scorers ready.")

## Section 2 — Reference Image Sets

4 sets of reference photos (one per character). Used only for ArcFace evaluation, not for generation.

In [ ]:
# Map reference filenames that actually exist in data/reference_faces/
REF_SETS = [
    {
        "hermione": "data/reference_faces/reference_hermione1.png",
        "daenerys": "data/reference_faces/reference_daenerys1.png",
    },
    {
        "hermione": "data/reference_faces/reference_hermione2.jpg",
        "daenerys": "data/reference_faces/reference_daenerys2.jpg",
    },
    {
        "hermione": "data/reference_faces/reference_hermione3.jpeg",
        "daenerys": "data/reference_faces/reference_daenerys3.jpg",
    },
    {
        "hermione": "data/reference_faces/reference_hermione4.jpeg",
        "daenerys": "data/reference_faces/reference_daenerys4.jpeg",
    },
]

# Validate that all files exist
for i, rs in enumerate(REF_SETS):
    for k, path in rs.items():
        if not os.path.exists(path):
            print(f"WARNING: ref_set {i+1} {k} missing: {path}")
        else:
            print(f"OK: ref_set {i+1} {k} -> {path}")

# Pre-extract ArcFace embeddings for all 4 reference sets
print("\nExtracting reference embeddings...")
REF_EMBED_SETS = []
for i, rs in enumerate(REF_SETS):
    embeds = {}
    for k, path in rs.items():
        img = Image.open(path).convert("RGB")
        emb = evaluator.extract_face_embedding(img)
        if emb is None:
            print(f"WARNING: No face detected in ref_set {i+1} {k}!")
        embeds[k] = emb
    REF_EMBED_SETS.append(embeds)
    print(f"ref_set {i+1}: hermione={embeds['hermione'] is not None}, "
          f"daenerys={embeds['daenerys'] is not None}")

print("\nAll reference embeddings extracted.")

## Section 3 — Pose Images

5 clearly distinct body poses:
1. Standing, arms at sides (default)
2. Both arms raised high overhead (celebration / V-shape)
3. Arms reaching toward each other (inner arms extended)
4. T-pose — arms spread fully horizontal
5. Asymmetric — each person: one arm up + opposite arm horizontal

In [ ]:
from PIL import ImageDraw
from face_aware_pose import (
    FACE_STRUCTURE_LINES, FACE_LINE_COLOR, FACE_LINE_WIDTH,
    KEYPOINT_RADIUS_FACE, KEYPOINT_RADIUS_BODY, _draw_dense_face,
    get_face_aware_target_keypoints,
)
from generate_pose import POSE_CONNECTIONS, LIMB_COLORS, KEYPOINT_COLOR

pose_dir = "data/pose_images/eval_poses"
os.makedirs(pose_dir, exist_ok=True)
pose_paths = [f"{pose_dir}/pose_{i+1}.png" for i in range(5)]

def make_custom_pose(kps1_norm, kps2_norm, width=1024, height=1024, dense_face=True):
    """Render a 2-person OpenPose skeleton from arbitrary normalized (x,y) keypoints."""
    img  = Image.new("RGB", (width, height), (0, 0, 0))
    draw = ImageDraw.Draw(img)
    face_idx = {0, 14, 15, 16, 17}
    for kps_norm in [kps1_norm, kps2_norm]:
        kps_px = [(int(x * width), int(y * height)) for x, y in kps_norm]
        for idx, (s, e) in enumerate(POSE_CONNECTIONS):
            draw.line([kps_px[s], kps_px[e]], fill=LIMB_COLORS[idx % len(LIMB_COLORS)], width=4)
        for a, b in FACE_STRUCTURE_LINES:
            draw.line([kps_px[a], kps_px[b]], fill=FACE_LINE_COLOR, width=FACE_LINE_WIDTH)
        if dense_face:
            _draw_dense_face(draw, kps_px)
        for i, (x, y) in enumerate(kps_px):
            r = KEYPOINT_RADIUS_FACE if i in face_idx else KEYPOINT_RADIUS_BODY
            draw.ellipse([x-r, y-r, x+r, y+r], fill=KEYPOINT_COLOR)
    return img

# Keypoint order (COCO-18):
#  0:nose 1:neck 2:r_shoulder 3:r_elbow 4:r_wrist
#  5:l_shoulder 6:l_elbow 7:l_wrist
#  8:r_hip 9:r_knee 10:r_ankle 11:l_hip 12:l_knee 13:l_ankle
#  14:r_eye 15:l_eye 16:r_ear 17:l_ear
# Person 1 centred at x≈0.30 (left half); Person 2 at x≈0.70 (right half)

# fmt: off
# ── Pose 1: Standing, arms at sides ───────────────────────────────────────
P1_A = [(0.30,0.18),(0.30,0.25),(0.24,0.27),(0.20,0.37),(0.18,0.45),
        (0.36,0.27),(0.40,0.37),(0.42,0.45),
        (0.26,0.45),(0.25,0.60),(0.25,0.75),(0.34,0.45),(0.35,0.60),(0.35,0.75),
        (0.28,0.16),(0.32,0.16),(0.25,0.17),(0.35,0.17)]
P1_B = [(0.70,0.18),(0.70,0.25),(0.64,0.27),(0.60,0.37),(0.58,0.45),
        (0.76,0.27),(0.80,0.37),(0.82,0.45),
        (0.66,0.45),(0.65,0.60),(0.65,0.75),(0.74,0.45),(0.75,0.60),(0.75,0.75),
        (0.68,0.16),(0.72,0.16),(0.65,0.17),(0.75,0.17)]

# ── Pose 2: Both arms raised high overhead (V / celebration shape) ─────────
P2_A = [(0.30,0.18),(0.30,0.25),(0.24,0.27),(0.17,0.13),(0.12,0.04),  # r_arm up
        (0.36,0.27),(0.43,0.13),(0.48,0.04),                            # l_arm up
        (0.26,0.45),(0.25,0.60),(0.25,0.75),(0.34,0.45),(0.35,0.60),(0.35,0.75),
        (0.28,0.16),(0.32,0.16),(0.25,0.17),(0.35,0.17)]
P2_B = [(0.70,0.18),(0.70,0.25),(0.64,0.27),(0.57,0.13),(0.52,0.04),  # r_arm up
        (0.76,0.27),(0.83,0.13),(0.88,0.04),                            # l_arm up
        (0.66,0.45),(0.65,0.60),(0.65,0.75),(0.74,0.45),(0.75,0.60),(0.75,0.75),
        (0.68,0.16),(0.72,0.16),(0.65,0.17),(0.75,0.17)]

# ── Pose 3: Arms reaching toward each other (inner arms extended) ──────────
P3_A = [(0.30,0.18),(0.30,0.25),(0.24,0.27),(0.20,0.37),(0.18,0.45),
        (0.36,0.27),(0.46,0.32),(0.55,0.34),                            # l_arm reaches right
        (0.26,0.45),(0.25,0.60),(0.25,0.75),(0.34,0.45),(0.35,0.60),(0.35,0.75),
        (0.28,0.16),(0.32,0.16),(0.25,0.17),(0.35,0.17)]
P3_B = [(0.70,0.18),(0.70,0.25),(0.64,0.27),(0.54,0.32),(0.45,0.34),  # r_arm reaches left
        (0.76,0.27),(0.80,0.37),(0.82,0.45),
        (0.66,0.45),(0.65,0.60),(0.65,0.75),(0.74,0.45),(0.75,0.60),(0.75,0.75),
        (0.68,0.16),(0.72,0.16),(0.65,0.17),(0.75,0.17)]

# ── Pose 4: T-pose — arms fully horizontal ────────────────────────────────
P4_A = [(0.30,0.18),(0.30,0.25),(0.24,0.27),(0.12,0.27),(0.02,0.27),  # r_arm right (away from center)
        (0.36,0.27),(0.48,0.27),(0.58,0.27),                            # l_arm left (toward center)
        (0.26,0.45),(0.25,0.60),(0.25,0.75),(0.34,0.45),(0.35,0.60),(0.35,0.75),
        (0.28,0.16),(0.32,0.16),(0.25,0.17),(0.35,0.17)]
P4_B = [(0.70,0.18),(0.70,0.25),(0.64,0.27),(0.52,0.27),(0.42,0.27),  # r_arm left (toward center)
        (0.76,0.27),(0.88,0.27),(0.98,0.27),                            # l_arm right (away from center)
        (0.66,0.45),(0.65,0.60),(0.65,0.75),(0.74,0.45),(0.75,0.60),(0.75,0.75),
        (0.68,0.16),(0.72,0.16),(0.65,0.17),(0.75,0.17)]

# ── Pose 5: Asymmetric — P1 right-arm-up + left-arm-horizontal; P2 mirror ─
P5_A = [(0.30,0.18),(0.30,0.25),(0.24,0.27),(0.17,0.13),(0.12,0.04),  # r_arm raised
        (0.36,0.27),(0.48,0.27),(0.58,0.27),                            # l_arm horizontal out
        (0.26,0.45),(0.25,0.60),(0.25,0.75),(0.34,0.45),(0.35,0.60),(0.35,0.75),
        (0.28,0.16),(0.32,0.16),(0.25,0.17),(0.35,0.17)]
P5_B = [(0.70,0.18),(0.70,0.25),(0.64,0.27),(0.52,0.27),(0.42,0.27),  # r_arm horizontal out
        (0.76,0.27),(0.83,0.13),(0.88,0.04),                            # l_arm raised
        (0.66,0.45),(0.65,0.60),(0.65,0.75),(0.74,0.45),(0.75,0.60),(0.75,0.75),
        (0.68,0.16),(0.72,0.16),(0.65,0.17),(0.75,0.17)]
# fmt: on

POSE_DEFS = [
    (P1_A, P1_B, "arms at sides"),
    (P2_A, P2_B, "both arms raised overhead"),
    (P3_A, P3_B, "arms reaching toward each other"),
    (P4_A, P4_B, "T-pose arms horizontal"),
    (P5_A, P5_B, "asymmetric one-up one-horizontal"),
]

# Always regenerate (PIL drawing, no GPU)
EVAL_POSE_IMAGES = []
for i, (kps1, kps2, desc) in enumerate(POSE_DEFS):
    img = make_custom_pose(kps1, kps2, dense_face=True)
    img.save(pose_paths[i])
    EVAL_POSE_IMAGES.append(img)
    print(f"Pose {i+1}: {desc}")
    display(img.resize((384, 384)))

target_keypoints = get_face_aware_target_keypoints(front_facing=True)
print("All 5 poses generated.")

## Section 4 — Prompt Variants

5 styles: formal/academic, royal/fantasy, modern casual, studio portrait, cinematic outdoor.

In [ ]:
# Each variant overrides visual_description per identity
# attention_phrases and LoRA config remain constant
PROMPT_VARIANTS = [
    {
        "name": "formal_academic",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, black Hogwarts school robe, ancient library background"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair, icy blonde, "
            "violet eyes, regal blue dress, ancient library background"
        ),
    },
    {
        "name": "royal_fantasy",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, elegant ceremonial robes, grand throne room with candlelight"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair, icy blonde, "
            "violet eyes, commanding royal gown, grand throne room with candlelight"
        ),
    },
    {
        "name": "modern_casual",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, casual blue jeans and white top, modern city street background"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair, icy blonde, "
            "violet eyes, white linen top and trousers, modern city street background"
        ),
    },
    {
        "name": "studio_portrait",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, professional studio lighting, clean white background, "
            "sharp focus close-up portrait"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair, icy blonde, "
            "violet eyes, professional studio lighting, clean white background, "
            "sharp focus close-up portrait"
        ),
    },
    {
        "name": "cinematic_outdoor",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, wind softly blowing hair, dramatic sunset cliff overlooking ocean"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair shimmering in sunlight, "
            "icy blonde, violet eyes, dramatic sunset cliff overlooking ocean"
        ),
    },
]

print(f"{len(PROMPT_VARIANTS)} prompt variants defined:")
for i, pv in enumerate(PROMPT_VARIANTS):
    print(f"  {i+1}. {pv['name']}")

## Section 5 — DARF v6 Config

Best configuration from v6 hair-refine experiments (strength=0.55).

In [ ]:
V6_TWO_STAGE_CONFIG = dict(
    layout_lora_scales={"hermione": 0.10, "daenerys": 0.30},
    identity_lora_scales={
        "hermione": {"down": 0.10, "mid": 0.20, "up": 0.45},
        "daenerys": {"down": 0.15, "mid": 0.55, "up": 1.50},
    },
    layout_ctrl_scale=1.00,
    identity_ctrl_scale=0.95,
    refine_strength=0.35,
    refine_steps=28,
    layout_steps=28,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    spatial_gate_block_floor={"down": 0.20, "mid": 0.05, "up": 0.00},
    use_bg_lock=True,
    bg_lock_ratio=0.40,
    bg_lock_padding=24,
    bg_lock_in_layout=True,
    bg_lock_in_identity=False,
    identity_regions=identity_regions,
)

FACE_REFINE_CONFIG = dict(
    refine_strength=0.40,
    crop_pad_ratio=0.5,
    refine_steps=28,
    feather=24,
)

HAIR_REFINE_CONFIG = dict(
    refine_strength=0.55,   # best from Bölüm 7
    refine_steps=28,
    feather=36,
    horiz_pad=0.6,
    up_pad=1.6,
    down_pad=0.2,
    lora_alpha=1.6,
)

print("DARF v6 config loaded.")
print(f"  hair_refine_strength : {HAIR_REFINE_CONFIG['refine_strength']}")
print(f"  stage2_refine_strength: {V6_TWO_STAGE_CONFIG['refine_strength']}")
print(f"  lora_up_hermione     : {V6_TWO_STAGE_CONFIG['identity_lora_scales']['hermione']['up']}")
print(f"  lora_up_daenerys     : {V6_TWO_STAGE_CONFIG['identity_lora_scales']['daenerys']['up']}")

## Section 6 — Output Setup

Create output directory, initialize CSV with existing run IDs for resume-safety.

In [ ]:
import pandas as pd
import time

OUT_DIR  = "data/results/darf_v6_100run"
CSV_PATH = f"{OUT_DIR}/darf_v6_100run_results.csv"
os.makedirs(OUT_DIR, exist_ok=True)

# Load existing run IDs for resume-safety
if os.path.exists(CSV_PATH):
    df_existing = pd.read_csv(CSV_PATH)
    existing_run_ids = set(df_existing["run_id"].tolist())
    print(f"Resuming: {len(existing_run_ids)}/100 runs already in CSV.")
else:
    existing_run_ids = set()
    print("Starting fresh — no existing CSV found.")

print(f"Output dir : {OUT_DIR}")
print(f"CSV path   : {CSV_PATH}")
print(f"Total runs : 4 ref × 5 pose × 5 prompt = 100")

## Section 7 — Main 100-Run Loop

Runs all 100 configurations. Resume-safe: skips if image + CSV row exist; eval-only if image exists but not in CSV.

In [ ]:
total = 4 * 5 * 5
done  = len(existing_run_ids)

for ref_idx in range(4):
    for pose_idx in range(5):
        for prompt_idx in range(5):

            run_id = ref_idx * 25 + pose_idx * 5 + prompt_idx
            seed   = 42 + run_id
            pv     = PROMPT_VARIANTS[prompt_idx]
            rs     = REF_SETS[ref_idx]

            final_path = f"{OUT_DIR}/run_{run_id:03d}_ref{ref_idx+1}_pose{pose_idx+1}_p{prompt_idx+1}.png"

            image_exists = os.path.exists(final_path)
            in_csv       = run_id in existing_run_ids

            if image_exists and in_csv:
                continue

            print(f"\n{'='*60}")
            print(f"Run {run_id:03d}/{total-1} | ref={ref_idx+1} pose={pose_idx+1} ({POSE_DEFS[pose_idx][2]}) "
                  f"| prompt={prompt_idx+1} ({pv['name']}) | seed={seed}")

            t0 = time.time()

            if image_exists and not in_csv:
                print("  [eval-only] image exists, re-evaluating...")
                final_img = Image.open(final_path).convert("RGB")
            else:
                identities["hermione"]["visual_description"] = pv["hermione_desc"]
                identities["daenerys"]["visual_description"] = pv["daenerys_desc"]

                _, stage2 = generate_two_stage(
                    pipe, identities, EVAL_POSE_IMAGES[pose_idx], seed=seed, **V6_TWO_STAGE_CONFIG
                )
                after_face = face_local_refine(
                    pipe, identities, stage2, face_scorer, identity_regions,
                    seed=seed, **FACE_REFINE_CONFIG
                )
                final_img = hair_local_refine(
                    pipe, identities, after_face, face_scorer, identity_regions,
                    seed=seed, **HAIR_REFINE_CONFIG
                )
                final_img.save(final_path)

            runtime = time.time() - t0

            ref_embeds = REF_EMBED_SETS[ref_idx]
            sim_h, sim_d = evaluator.detect_and_assign_faces(
                final_img, ref_embeds["hermione"], ref_embeds["daenerys"]
            )
            n_faces = detected_face_count(final_img)

            try:
                pose_err = evaluator.pose_keypoint_error(final_img, EVAL_POSE_IMAGES[pose_idx])
            except Exception as ex:
                print(f"  [pose eval error]: {ex}")
                pose_err = -1.0

            row = {
                "run_id":               run_id,
                "ref_idx":              ref_idx + 1,
                "pose_idx":             pose_idx + 1,
                "prompt_idx":           prompt_idx + 1,
                "prompt_name":          pv["name"],
                "seed":                 seed,
                "ref_hermione_path":    rs["hermione"],
                "ref_daenerys_path":    rs["daenerys"],
                "pose_path":            pose_paths[pose_idx],
                "final_image_path":     final_path,
                "arcface_hermione":     round(float(sim_h), 4),
                "arcface_daenerys":     round(float(sim_d), 4),
                "arcface_mean":         round((float(sim_h) + float(sim_d)) / 2, 4),
                "face_count":           n_faces,
                "extra_person_flag":    int(n_faces > 2),
                "pose_metric":          round(float(pose_err), 4),
                "runtime_seconds":      round(runtime, 1),
                "hair_refine_strength": HAIR_REFINE_CONFIG["refine_strength"],
                "stage2_refine_strength": V6_TWO_STAGE_CONFIG["refine_strength"],
                "lora_up_hermione":     V6_TWO_STAGE_CONFIG["identity_lora_scales"]["hermione"]["up"],
                "lora_up_daenerys":     V6_TWO_STAGE_CONFIG["identity_lora_scales"]["daenerys"]["up"],
            }

            existing_run_ids.add(run_id)
            done += 1

            write_header = not os.path.exists(CSV_PATH)
            pd.DataFrame([row]).to_csv(CSV_PATH, mode="a", header=write_header, index=False)

            print(f"  arc_h={sim_h:.3f}  arc_d={sim_d:.3f}  mean={row['arcface_mean']:.3f}  "
                  f"faces={n_faces}  extra={row['extra_person_flag']}  "
                  f"pose_err={pose_err:.3f}  {runtime:.0f}s  [{done}/{total} done]")

            if done % 10 == 0:
                display(final_img.resize((384, 384)))

print(f"\n{'='*60}")
print(f"Loop complete. {done}/{total} runs done. CSV at {CSV_PATH}")

## Section 8 — Results Analysis

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df)}")
df.head(10)

In [ ]:
numeric_cols = ["arcface_hermione", "arcface_daenerys", "arcface_mean",
                "face_count", "extra_person_flag", "pose_metric", "runtime_seconds"]

print("=== Overall Statistics ===")
print(df[numeric_cols].describe().T[["mean", "std", "min", "50%", "max"]].round(4))

print(f"\n=== Summary ===")
print(f"Total runs            : {len(df)}")
print(f"ArcFace mean ± std    : {df['arcface_mean'].mean():.4f} ± {df['arcface_mean'].std():.4f}")
print(f"ArcFace Hermione      : {df['arcface_hermione'].mean():.4f} ± {df['arcface_hermione'].std():.4f}")
print(f"ArcFace Daenerys      : {df['arcface_daenerys'].mean():.4f} ± {df['arcface_daenerys'].std():.4f}")
print(f"Both >0.25 success    : {((df['arcface_hermione']>0.25)&(df['arcface_daenerys']>0.25)).sum()}/{len(df)}")
print(f"Extra person rate     : {df['extra_person_flag'].mean()*100:.1f}%")
print(f"Exactly 2 faces rate  : {(df['face_count']==2).mean()*100:.1f}%")
print(f"Mean runtime / Total  : {df['runtime_seconds'].mean():.0f}s / {df['runtime_seconds'].sum()/3600:.1f}h")

In [ ]:
print("=== By Prompt Style ===")
grp_prompt = df.groupby("prompt_name").agg(
    arc_mean=("arcface_mean", "mean"),
    arc_h=("arcface_hermione", "mean"),
    arc_d=("arcface_daenerys", "mean"),
    extra_rate=("extra_person_flag", "mean"),
    pose_err=("pose_metric", "mean"),
    n=("run_id", "count"),
).round(4)
print(grp_prompt.to_string())

In [ ]:
print("=== By Pose ===")
pose_labels = {i+1: d for i, (_, _, d) in enumerate(POSE_DEFS)}
grp_pose = df.groupby("pose_idx").agg(
    arc_mean=("arcface_mean", "mean"),
    arc_h=("arcface_hermione", "mean"),
    arc_d=("arcface_daenerys", "mean"),
    extra_rate=("extra_person_flag", "mean"),
    pose_err=("pose_metric", "mean"),
    n=("run_id", "count"),
).round(4)
grp_pose.index = grp_pose.index.map(lambda x: f"{x}: {pose_labels.get(x, '')}")
print(grp_pose.to_string())

In [ ]:
print("=== By Reference Set ===")
grp_ref = df.groupby("ref_idx").agg(
    arc_mean=("arcface_mean", "mean"),
    arc_h=("arcface_hermione", "mean"),
    arc_d=("arcface_daenerys", "mean"),
    extra_rate=("extra_person_flag", "mean"),
    n=("run_id", "count"),
).round(4)
print(grp_ref.to_string())

In [ ]:
cols = ["run_id", "ref_idx", "pose_idx", "prompt_name",
        "arcface_hermione", "arcface_daenerys", "arcface_mean", "face_count", "final_image_path"]

print("=== Top 5 Runs ===")
top5 = df.nlargest(5, "arcface_mean")[cols]
print(top5.to_string(index=False))

print("\n=== Bottom 5 Runs ===")
print(df.nsmallest(5, "arcface_mean")[cols].to_string(index=False))

print("\nTop 5 images:")
for _, row in top5.iterrows():
    path = row["final_image_path"]
    if os.path.exists(path):
        print(f"run_{int(row['run_id']):03d} | {row['prompt_name']} | "
              f"arc_h={row['arcface_hermione']:.3f} arc_d={row['arcface_daenerys']:.3f}")
        display(Image.open(path).resize((384, 384)))

## Section 9 — Save Summary CSVs + ZIP

In [ ]:
grp_prompt.to_csv(f"{OUT_DIR}/summary_by_prompt.csv")
grp_pose.to_csv(f"{OUT_DIR}/summary_by_pose.csv")
grp_ref.to_csv(f"{OUT_DIR}/summary_by_ref.csv")
print(f"Saved summary CSVs to {OUT_DIR}/")

import shutil
from IPython.display import FileLink
zip_path = "/workspace/darf_v6_100run_results"
shutil.make_archive(zip_path, "zip", OUT_DIR)
print(f"ZIP ready: {zip_path}.zip")
FileLink(f"{zip_path}.zip")